### SVM implemented in JAX

In this notebook we will aim to clearly explain the impact of Kernels in Support Vector Machines implemented in JAX

### Introduction to Support Vector Machines

A support vector machine is a supervised learning model, that finds the optimal hyperplane separating two or more classes by maximizing the margin.

Given a training set, a SVM decision function is defined as : 
$ f(x) = w^T x + b $

And each point is classified by :
$ \hat{y} = sign(f(x)) $

and by its covariance fonction $$ k(x,x') = \mathbb{E}[(f(x) - m(x))((f(x') - m(x')))] = 0 $$

We will consider a GP as $$ \mathbb f(x) ∼ GP(m(x), k(x,x')) $$

### Class Definition
We have a SVM class that considers the important elements that will intervene in our differents concepts, such as the Kernel covariance matrix, the error term, the weights and the bias

In [ ]:
import jax.numpy as jnp
import sys
from jax import jit
from functools import partial
from kernax import LinearKernel, SEKernel
import jax
import optax



def __init__(self, C = 1.0, kernel = None):
        # C = error term
        self.C = C
        self.w = 0
        self.b = 0
        self.kernel = None
        self.X_train = None  
        self.y_train = None  
        self.alpha = None
        self.kernel_fn = kernel

### Hinge loss 
Basically the Hinge Loss function is the cost function for the primal case of the SVM.
It's defined by : 
$L(w, b) = \frac{1}{2}\|w\|^2 + C \cdot \frac{1}{n}\sum_{i=1}^{n} \max(0, 1 - y_i(w^T x_i + b)) $

- $w$ corresponds to the weights
- $C$ is the regularization parameter that control the trade-off between the margin and potentiel errors of classification

In [ ]:
# Hinge Loss Function / Calculation
@partial(jit, static_argnums=(0,))
def hingeloss(self, w, b, x, y):
    # Regularizer term
     reg = 0.5 * jnp.sum(w * w)  
    
    
    # We calculate all the scores in one time, so we don't use a for loop
     scores = jnp.dot(x, w) + b
     
    # hinge loss for all elements of our sample
     hinge_losses = jnp.maximum(0, 1 - y * scores)
    
    # mean of all elements of our sample
     avg_hinge = jnp.mean(hinge_losses)
    
    # total loss
     total_loss = reg + self.C * avg_hinge
    
     return total_loss

### Dual Loss
Basically it's just an optimization of the hinge loss, for the coefficients $a$ 
$\mathcal{L}(\alpha) = \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j) - \sum_i \alpha_i + C \sum_i \max(0, \alpha_i - C)^2 + C \sum_i \max(0, -\alpha_i)^2$

The constraint in the dual loss function is that the coefficients $a$ should always be inferior to the regularization parameter


In [ ]:
def fit(self, X, Obs, num_iters=1000, learning_rate=0.1, progress=False):
      
        X = jnp.asarray(X)
        Obs = jnp.asarray(Obs)
        if Obs.ndim == 1:
            Obs = Obs[:, None]  #Converting to (n,1) if needed

        #Normalizing the data if needed, we make the hypothesis that the mean is null
        #, but if the mean isn't then we need to normalize the Obs

        if self.NormalizeObs:
            ObsMean = jnp.mean(Obs, axis=0)
            ObsSTD = jnp.std(Obs, axis=0)
            ObsSTD = jnp.where(ObsSTD == 0, 1.0,ObsSTD)
            Obs_norm = (Obs -ObsMean) /ObsSTD
        else:
           ObsMean = jnp.zeros(Obs.shape[1:])
           ObsSTD = jnp.ones_like(ObsMean)
           Obs_norm = Obs

        #Simple loss function (log likelihood that we will minimize)
        def loss_fn(Kernel):
            return -self.marginal_log_likelihood(Kernel, X, Obs_norm, self.Alpha)

        
        optimizer = optax.adam(learning_rate)
        opt_state = optimizer.init(self.Kernel)

        #Classical optimization of the hyperparameters of the kernel
        @jit
        def step(Kernel, opt_state):
            loss, grads = value_and_grad(loss_fn)(Kernel)
            updates, opt_state = optimizer.update(grads, opt_state, Kernel)
            Kernel = optax.apply_updates(Kernel, updates)
            return Kernel, opt_state, loss

        #Optimization loop that give us losses
        Kernel = self.Kernel
        for i in range(num_iters):
            Kernel, opt_state, loss = step(Kernel, opt_state)

        #Update of the kernel optimized(the hyparmaters)
        self.Kernel = Kernel

        #Calculating the new values with optimized parameters
        K = self.Kernel(X,X)
        K = K + self.Alpha * jnp.eye(len(X))
        L = jnp.linalg.cholesky(K)
        Alpha_ = jax.scipy.linalg.cho_solve((L, True), Obs_norm)

        #Updating the state
        self.TrainingData = X
        self.Obs = Obs_norm
        self.L_ = L
        self.Alpha_ = Alpha_
        self.ObsMean =ObsMean
        self.ObsSTD =ObsSTD

        return self

### Predict function 

The predict function will give us the posterior mean and covariance for a set of prediction points, which will be unnormalized(if the targets weren't normalized, nothing is done)

We will compute the value of the posterior mean :
$$
\mu_* = K(X_*, X) \bigl(K(X,X) + \alpha I\bigr)^{-1} y
$$

- $K$ corresponds to the gram matrix
- $ X_*$ corresponds to the prediction points
- $\alpha$ corresponds to the noise variance
- $y$ corresponds to the targets (normalized or not)

and we will compute the value of the posterior covariance

$$
\Sigma_* = K(X_*, X_*) - K(X_*, X) \bigl(K(X,X) + \alpha I\bigr)^{-1} K(X, X_*)
$$

- $K$ corresponds to the gram matrix
- $X_* $ corresponds to the prediction points
- $\alpha $ corresponds to the noise variance


In [ ]:
@partial(jit, static_argnames=['Return_std', 'Return_cov'])
def predict(self, X, Return_std=False, Return_cov=False):
     X = jnp.asarray(X)

     #Calculation of the diagonal(variance for each test random variable)
     Diag_fn = jax.vmap(lambda x: self.Kernel(x, x))

     if self.TrainingData is None:
        #If the model wasn't trained we calculate the prior(basically it's just applying,
        # The kernel for each pair of test points)
        n_targets = 1 if self.ObsMean is None else self.ObsMean.shape[0]
        ObsMean = jnp.zeros((X.shape[0], n_targets)).squeeze()
        if Return_cov:
            Obs_cov = self.Kernel(X,X)
            return ObsMean, Obs_cov
        elif Return_std:
            Obs_var = Diag_fn(X)   
            return ObsMean, jnp.sqrt(Obs_var)
        else:
            return ObsMean
     else:
        # If the model is trained then we apply the formulas to find the new distributions
        # so basically we want to find the updated mean and cov matrix
        K_trans = self.Kernel(X, self.TrainingData)
        ObsMean = K_trans @ self.Alpha_
        ObsMean = self.ObsSTD *ObsMean + self.ObsMean
        if ObsMean.ndim > 1 and ObsMean.shape[1] == 1:
           ObsMean =ObsMean.squeeze(1)

        if Return_cov:
            v = jax.scipy.linalg.solve_triangular(self.L_, K_trans.T, lower=True)
            Obs_cov = self.Kernel(X,X) - v.T @ v
            Obs_cov = jnp.outer(Obs_cov, self.ObsSTD**2).reshape(*Obs_cov.shape, -1)
            if Obs_cov.shape[-1] == 1:
                Obs_cov = Obs_cov.squeeze(-1)
            return ObsMean, Obs_cov
        elif Return_std:
            v = jax.scipy.linalg.solve_triangular(self.L_, K_trans.T, lower=True)
            Obs_var = Diag_fn(X) - jnp.einsum('ij,ji->i', v.T, v)
            Obs_var = Obs_var * self.ObsSTD**2
            return ObsMean, jnp.sqrt(Obs_var)
        else:
            return ObsMean